# NovaTel | Laboratorium 2
## Czyszczenie, transformacja i kontrola jakości danych

**Materiały:** `dane_novatel_lab2_surowe.csv` i karta pracy. Reguły biznesowe oraz miejsce na zapis decyzji znajdują się w karcie.

Wykonuj zadania kolejno. Nie zmieniaj pliku źródłowego. Wszystkie obliczenia i decyzje realizuj samodzielnie.

## Przygotowanie środowiska

Umieść plik CSV w tym samym folderze co notebook. Poniższa komórka tylko wczytuje dane i tworzy kopię roboczą.

In [184]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_PATH = Path("dane_novatel_lab2_surowe.csv")
raw = pd.read_csv(DATA_PATH)
df = raw.copy()
raw.head()


,customer_id,snapshot_date,wiek,płeć,województwo,wielkość_miejscowości,typ_umowy,staż_miesiące,opłata_miesięczna_pln,średnie_użycie_h_mies,...,kontakty_z_obsługą_3m,reklamacje_6m,satysfakcja_1_5,zgoda_marketingowa,kanał_pozyskania,rabat_proc,dni_od_ostatniego_logowania,dochód_miesięczny_pln,odejście_90d,data_zamknięcia_konta
0,NT10234,2026-10-01,38.0,M,Mazowieckie,wieś,miesięczna,9,96.23,32.3,...,1,0,4.0,tak,online,24.0,3,13700.0,tak,2026-12-04
1,NT10110,2026-10-01,45.0,inna/nie podano,Mazowieckie,średnie miasto,12 miesięcy,67,100.62,63.9,...,1,3,4.0,nie,salon,13.0,9,18840.0,nie,NaN
2,NT10248,2026-10-01,36.0,M,Pomorskie,małe miasto,12 miesięcy,16,115.73,28.7,...,2,0,3.0,tak,salon,7.0,6,4220.0,nie,NaN
3,NT10009,2026-10-01,34.0,M,Dolnośląskie,średnie miasto,12 miesięcy,10,110.10,39.0,...,1,0,3.0,nie,telefon,4.0,4,9020.0,nie,NaN
4,NT10093,2026-10-01,34.0,M,Mazowieckie,duże miasto,miesięczna,32,117.50,38.8,...,1,0,4.0,nie,online,15.0,5,10260.0,tak,2026-12-26


## 1. Kontrola wejścia

Sprawdź wymiary zbioru, nazwy i typy kolumn, braki danych, unikalność `customer_id`, pełne duplikaty oraz udział odejść. Uzupełnij tabelę w karcie pracy.

In [185]:
# wymiary i typy kolumn
print("Wymiary:", raw.shape)
print(raw.dtypes)

# braki danych w każdej kolumnie
print(raw.isna().sum())

# unikalność identyfikatora i duplikaty
print("customer_id unikalne:", raw["customer_id"].is_unique)
print("Powtórzone customer_id:", raw["customer_id"].duplicated().sum())
print("Pełne duplikaty:", raw.duplicated().sum())

# udział odejść
print(raw["odejście_90d"].value_counts(normalize=True).round(3))

# słownik z miarami kontrolnymi (stan startowy)
stan_startowy = {
    "wiersze": len(raw),
    "kolumny": raw.shape[1],
    "unikalne_id": raw["customer_id"].nunique(),
    "duplikaty": int(raw.duplicated().sum()),
}
print(stan_startowy)

Wymiary: (325, 21)
customer_id                        str
snapshot_date                      str
wiek                           float64
płeć                               str
województwo                        str
wielkość_miejscowości              str
typ_umowy                          str
staż_miesiące                    int64
opłata_miesięczna_pln          float64
średnie_użycie_h_mies          float64
opóźnienia_płatności_6m          int64
kontakty_z_obsługą_3m            int64
reklamacje_6m                    int64
satysfakcja_1_5                float64
zgoda_marketingowa                 str
kanał_pozyskania                   str
rabat_proc                     float64
dni_od_ostatniego_logowania      int64
dochód_miesięczny_pln          float64
odejście_90d                       str
data_zamknięcia_konta              str
dtype: object
customer_id                      0
snapshot_date                    0
wiek                            12
płeć                             0
wojewódz

## 2. Rejestr decyzji

Na podstawie reguł z karty pracy zaplanuj, jak rozwiążesz wykryte problemy. Uzupełniaj rejestr decyzji w trakcie kolejnych etapów.

## 3. Duplikaty i identyfikatory

Porównaj powtórzone identyfikatory z pełnymi duplikatami. Wyświetl problematyczne rekordy, podejmij decyzję o deduplikacji i sprawdź wynik.

In [186]:
# reset kopii roboczej (pozwala bezpiecznie uruchamiać ten krok ponownie)
df = raw.copy()

# dziennik zmian (lista + append)
historia = []

# obejrzenie rekordów z powtórzonym id
podejrzane = df[df["customer_id"].duplicated(keep=False)]
print(podejrzane.sort_values("customer_id"))

# powtórzone id = pełne duplikaty? (5 i 5)
print(df["customer_id"].duplicated().sum(), df.duplicated().sum())

# usunięcie pełnych duplikatów
przed = len(df)
df = df.drop_duplicates().reset_index(drop=True)
historia.append(["deduplikacja", przed - len(df), "usunięto pełne kopie rekordów"])

# kontrola wyniku
print(len(df))
assert df["customer_id"].is_unique

    customer_id snapshot_date  wiek płeć    województwo wielkość_miejscowości  \
22      NT10046    2026-10-01  38.0    K   Dolnośląskie        średnie miasto   
60      NT10046    2026-10-01  38.0    K   Dolnośląskie        średnie miasto   
192     NT10142    2026-10-01  32.0    K  Wielkopolskie           duże miasto   
318     NT10142    2026-10-01  32.0    K  Wielkopolskie           duże miasto   
14      NT10191    2026-10-01  64.0    M  Wielkopolskie        średnie miasto   
307     NT10191    2026-10-01  64.0    M  Wielkopolskie        średnie miasto   
228     NT10216    2026-10-01  50.0    K   Dolnośląskie           duże miasto   
259     NT10216    2026-10-01  50.0    K   Dolnośląskie           duże miasto   
0       NT10234    2026-10-01  38.0    M    Mazowieckie                  wieś   
297     NT10234    2026-10-01  38.0    M    Mazowieckie                  wieś   

       typ_umowy  staż_miesiące  opłata_miesięczna_pln  średnie_użycie_h_mies  \
22    miesięczna           

## 4. Standaryzacja kategorii i dat

Oczyść niejednolite zapisy w kolumnach tekstowych, nie gubiąc wartości brakujących. Uporządkuj województwa i kanały pozyskania. Skonwertuj daty na właściwy typ. Skontroluj kategorie przed i po zmianach.

In [187]:
kolumny_tekstowe = ["płeć", "województwo", "wielkość_miejscowości", "typ_umowy",
                    "zgoda_marketingowa", "kanał_pozyskania", "odejście_90d"]

# PRZED
for kolumna in ["województwo", "kanał_pozyskania", "zgoda_marketingowa", "odejście_90d"]:
    print(kolumna, "PRZED:", df[kolumna].value_counts(dropna=False).to_dict())

# spacje i małe litery (braki zostają brakami)
for kolumna in kolumny_tekstowe:
    df[kolumna] = df[kolumna].astype("string").str.strip().str.lower()

# województwa: pierwsza duża litera
df["województwo"] = df["województwo"].str.capitalize()

# kanały: mapowanie (wartość spoza słownika = brak, więc słownik ma wszystkie warianty)
mapa_kanalow = {"online": "online", "salon": "salon", "telefon": "telefon", "partner": "partner"}
df["kanał_pozyskania"] = df["kanał_pozyskania"].map(mapa_kanalow)

# PO
for kolumna in ["województwo", "kanał_pozyskania", "zgoda_marketingowa", "odejście_90d"]:
    print(kolumna, "PO:", df[kolumna].value_counts(dropna=False).to_dict())

# daty
df["snapshot_date"] = pd.to_datetime(df["snapshot_date"], errors="coerce")
df["data_zamknięcia_konta"] = pd.to_datetime(df["data_zamknięcia_konta"], errors="coerce")
print(df[["snapshot_date", "data_zamknięcia_konta"]].dtypes)
print("Błędne daty (NaT) w snapshot_date:", df["snapshot_date"].isna().sum())

# cel w postaci 0/1
df["odejście_90d_bin"] = df["odejście_90d"].map({"nie": 0, "tak": 1})
print("Braki w odejście_90d_bin:", df["odejście_90d_bin"].isna().sum())

województwo PRZED: {'Mazowieckie': 77, 'Śląskie': 62, 'Dolnośląskie': 55, 'Wielkopolskie': 46, 'Małopolskie': 37, 'Pomorskie': 34, 'Pomorskie ': 3, 'MAZOWIECKIE': 3, 'dolnośląskie': 3}
kanał_pozyskania PRZED: {'online': 148, 'salon': 67, 'telefon': 49, 'partner': 41, nan: 7, 'Online': 3, 'telefon ': 3, 'SALON': 2}
zgoda_marketingowa PRZED: {'tak': 227, 'nie': 93}
odejście_90d PRZED: {'nie': 292, 'tak': 28}
województwo PO: {'Mazowieckie': 80, 'Śląskie': 62, 'Dolnośląskie': 58, 'Wielkopolskie': 46, 'Pomorskie': 37, 'Małopolskie': 37}
kanał_pozyskania PO: {'online': 151, 'salon': 69, 'telefon': 52, 'partner': 41, nan: 7}
zgoda_marketingowa PO: {'tak': 227, 'nie': 93}
odejście_90d PO: {'nie': 292, 'tak': 28}
snapshot_date            datetime64[us]
data_zamknięcia_konta    datetime64[us]
dtype: object
Błędne daty (NaT) w snapshot_date: 0
Braki w odejście_90d_bin: 0


## 5. Wartości niepoprawne

Zastosuj kontrole zakresów wymienione w karcie. Policz wykryte błędy i popraw je zgodnie z regułami właściciela danych. Nie usuwaj całych rekordów bez uzasadnienia.

In [188]:
# maski błędów (notna() odróżnia błąd od braku danych)
zly_wiek        = df["wiek"].notna() & ~df["wiek"].between(18, 100)
zla_oplata      = df["opłata_miesięczna_pln"] <= 0
zla_satysfakcja = df["satysfakcja_1_5"].notna() & ~df["satysfakcja_1_5"].between(1, 5)
zly_rabat       = ~df["rabat_proc"].between(0, 100)
zly_staz        = df["staż_miesiące"] < 0
zle_uzycie      = df["średnie_użycie_h_mies"] < 0
zly_dochod      = df["dochód_miesięczny_pln"] <= 0

print("wiek:", zly_wiek.sum(), "| opłata:", zla_oplata.sum(),
      "| satysfakcja:", zla_satysfakcja.sum(),
      "| rabat:", zly_rabat.sum(), "| staż:", zly_staz.sum(),
      "| użycie:", zle_uzycie.sum(), "| dochód:", zly_dochod.sum())
print("staż = 0 (poprawny, rzadki):", (df["staż_miesiące"] == 0).sum())

# podgląd błędnych wartości
print(df.loc[zly_wiek | zla_oplata | zla_satysfakcja,
             ["customer_id", "wiek", "opłata_miesięczna_pln", "satysfakcja_1_5"]])

# poprawa: błąd -> brak (nie usuwamy rekordów)
df.loc[zly_wiek, "wiek"] = np.nan
df.loc[zla_oplata, "opłata_miesięczna_pln"] = np.nan
df.loc[zla_satysfakcja, "satysfakcja_1_5"] = np.nan

historia.append(["wiek poza 18-100", int(zly_wiek.sum()), "zamieniono na brak"])
historia.append(["opłata <= 0", int(zla_oplata.sum()), "zamieniono na brak"])
historia.append(["satysfakcja poza 1-5", int(zla_satysfakcja.sum()), "zamieniono na brak"])

# podgląd dziennika zmian
print(pd.DataFrame(historia, columns=["krok", "liczba", "opis"]))

wiek: 1 | opłata: 1 | satysfakcja: 1 | rabat: 0 | staż: 0 | użycie: 0 | dochód: 0
staż = 0 (poprawny, rzadki): 1
    customer_id   wiek  opłata_miesięczna_pln  satysfakcja_1_5
16      NT10146  129.0                 100.81              5.0
201     NT10089   47.0                 -49.99              4.0
262     NT10296   18.0                 145.11              7.0
                   krok  liczba                           opis
0          deduplikacja       5  usunięto pełne kopie rekordów
1      wiek poza 18-100       1             zamieniono na brak
2           opłata <= 0       1             zamieniono na brak
3  satysfakcja poza 1-5       1             zamieniono na brak


## 6. Braki danych

Sprawdź braki po oznaczeniu wartości niepoprawnych. **Przed imputacją** dodaj znaczniki: `brak_wieku`, `brak_satysfakcji`, `brak_dochodu`, `brak_kanału`. Następnie wybierz i zastosuj strategie obsługi braków. Zapisz uzasadnienia w karcie.

In [189]:
# braki PO oznaczeniu błędów
print(df.isna().sum())

# 1) FLAGI – muszą powstać PRZED fillna()
df["brak_wieku"]       = df["wiek"].isna().astype(int)
df["brak_satysfakcji"] = df["satysfakcja_1_5"].isna().astype(int)
df["brak_dochodu"]     = df["dochód_miesięczny_pln"].isna().astype(int)
df["brak_kanału"]      = df["kanał_pozyskania"].isna().astype(int)
print(df[["brak_wieku", "brak_satysfakcji", "brak_dochodu", "brak_kanału"]].sum())

# 2) imputacja globalna (mediana)
df["wiek"] = df["wiek"].fillna(df["wiek"].median())
df["satysfakcja_1_5"] = df["satysfakcja_1_5"].fillna(df["satysfakcja_1_5"].median())

# 3) imputacja w grupach (groupby + transform)
mediana_dochodu = df.groupby("wielkość_miejscowości")["dochód_miesięczny_pln"].transform("median")
df["dochód_miesięczny_pln"] = df["dochód_miesięczny_pln"].fillna(mediana_dochodu)

mediana_oplaty = df.groupby("typ_umowy")["opłata_miesięczna_pln"].transform("median")
df["opłata_miesięczna_pln"] = df["opłata_miesięczna_pln"].fillna(mediana_oplaty)

# 4) kategoria
df["kanał_pozyskania"] = df["kanał_pozyskania"].fillna("nieznany")

# kontrola (data_zamknięcia_konta wyłączona: brak daty = klient nie odszedł)
print("Braki po imputacji:", df.drop(columns=["data_zamknięcia_konta"]).isna().sum().sum())

customer_id                      0
snapshot_date                    0
wiek                            13
płeć                             0
województwo                      0
wielkość_miejscowości            0
typ_umowy                        0
staż_miesiące                    0
opłata_miesięczna_pln            1
średnie_użycie_h_mies            0
opóźnienia_płatności_6m          0
kontakty_z_obsługą_3m            0
reklamacje_6m                    0
satysfakcja_1_5                 19
zgoda_marketingowa               0
kanał_pozyskania                 7
rabat_proc                       0
dni_od_ostatniego_logowania      0
dochód_miesięczny_pln           28
odejście_90d                     0
data_zamknięcia_konta          292
odejście_90d_bin                 0
dtype: int64
brak_wieku          13
brak_satysfakcji    19
brak_dochodu        28
brak_kanału          7
dtype: int64
Braki po imputacji: 0


## 7. Cechy pochodne

Zaprojektuj i wylicz co najmniej trzy nowe cechy o sensie gospodarczym. Możesz wykorzystać propozycje z karty pracy. Zwróć uwagę na wartości brakujące, dzielenie przez zero i dostępność informacji w czasie prognozy.

In [190]:
# 1) staż w latach
df["staż_lata"] = (df["staż_miesiące"] / 12).round(2)

# 2) kontakty z obsługą + reklamacje (sygnał niezadowolenia)
df["kontakt_reklamacja_razem"] = df["kontakty_z_obsługą_3m"] + df["reklamacje_6m"]

# 3) aktywność: logowanie w ostatnich 7 dniach przed migawką
df["aktywny_ostatnie_7d"] = (df["dni_od_ostatniego_logowania"] <= 7).astype(int)

# 4) udział opłaty w dochodzie (%) – dochód > 0, więc brak dzielenia przez zero
df["udział_opłaty_w_dochodzie_proc"] = (100 * df["opłata_miesięczna_pln"] / df["dochód_miesięczny_pln"]).round(2)

# 5) opłata po rabacie (PLN)
df["opłata_po_rabacie_pln"] = (df["opłata_miesięczna_pln"] * (1 - df["rabat_proc"] / 100)).round(2)

# 6) koszt za godzinę użycia (mianownik > 0, żeby nie dzielić przez zero)
df["koszt_za_godzinę_pln"] = (df["opłata_miesięczna_pln"] /
                              df["średnie_użycie_h_mies"].where(df["średnie_użycie_h_mies"] > 0)).round(2)

# (odejście_90d_bin – cel 0/1 – utworzony w kroku 4; to NIE jest predyktor)

nowe = ["staż_lata", "kontakt_reklamacja_razem", "aktywny_ostatnie_7d",
        "udział_opłaty_w_dochodzie_proc", "opłata_po_rabacie_pln", "koszt_za_godzinę_pln"]
print(df[nowe].describe().T)
print("Wartości inf:", np.isinf(df[nowe]).sum().sum())
print("Braki w nowych cechach:", df[nowe].isna().sum().sum())

                                count       mean        std    min      25%  \
staż_lata                       320.0   2.328344   1.464116   0.00   1.2500   
kontakt_reklamacja_razem        320.0   1.828125   1.336194   0.00   1.0000   
aktywny_ostatnie_7d             320.0   0.612500   0.487942   0.00   0.0000   
udział_opłaty_w_dochodzie_proc  320.0   1.288687   0.682796   0.15   0.8275   
opłata_po_rabacie_pln           320.0  86.945937  25.566946  29.05  69.3425   
koszt_za_godzinę_pln            320.0   5.424500  20.801279   0.52   1.9600   

                                   50%       75%     max  
staż_lata                        2.040    3.2500    7.33  
kontakt_reklamacja_razem         2.000    3.0000    7.00  
aktywny_ostatnie_7d              1.000    1.0000    1.00  
udział_opłaty_w_dochodzie_proc   1.185    1.6025    6.04  
opłata_po_rabacie_pln           85.750  104.6300  168.83  
koszt_za_godzinę_pln             2.585    3.6125  257.20  
Wartości inf: 0
Braki w nowych ce

## 8. Pola do modelowania i audytu

Wybierz kandydatów na predyktory i uzasadnij wyłączenia. Oddziel pola identyfikacyjne, informację dostępną dopiero po zdarzeniu oraz pola, których użycie wymaga dodatkowej oceny. Zapisz listę kandydatów w zmiennej `candidate_features`.

In [191]:
# pola identyfikacyjne / techniczne – nie są predyktorami
pola_id = ["customer_id", "snapshot_date"]

# wyciek informacji: znane dopiero po zdarzeniu (lub sam cel)
pola_po_zdarzeniu = ["data_zamknięcia_konta", "odejście_90d", "odejście_90d_bin"]

# pola wrażliwe – wymagają dodatkowej oceny, zostają w zbiorze do audytu (fairness)
pola_audytowe = ["płeć", "wiek", "województwo", "wielkość_miejscowości",
                 "dochód_miesięczny_pln", "brak_wieku", "brak_dochodu",
                 "udział_opłaty_w_dochodzie_proc"]

wykluczone = set(pola_id + pola_po_zdarzeniu + pola_audytowe)
candidate_features = [k for k in df.columns if k not in wykluczone]

print("Kandydaci na predyktory:", len(candidate_features))
print(candidate_features)
print("Do audytu:", pola_audytowe)

assert "data_zamknięcia_konta" not in candidate_features
assert "odejście_90d_bin" not in candidate_features

Kandydaci na predyktory: 19
['typ_umowy', 'staż_miesiące', 'opłata_miesięczna_pln', 'średnie_użycie_h_mies', 'opóźnienia_płatności_6m', 'kontakty_z_obsługą_3m', 'reklamacje_6m', 'satysfakcja_1_5', 'zgoda_marketingowa', 'kanał_pozyskania', 'rabat_proc', 'dni_od_ostatniego_logowania', 'brak_satysfakcji', 'brak_kanału', 'staż_lata', 'kontakt_reklamacja_razem', 'aktywny_ostatnie_7d', 'opłata_po_rabacie_pln', 'koszt_za_godzinę_pln']
Do audytu: ['płeć', 'wiek', 'województwo', 'wielkość_miejscowości', 'dochód_miesięczny_pln', 'brak_wieku', 'brak_dochodu', 'udział_opłaty_w_dochodzie_proc']


## 9. Powtarzalny proces i testy jakości

Zamknij wykonane operacje w funkcji `clean_novatel(raw_df)`, która nie modyfikuje argumentu i zwraca oczyszczoną kopię. Uruchom ją na pierwotnym zbiorze i zapisz wynik w `cleaned`. Następnie napisz własne testy warunków z karty pracy.

In [192]:
def clean_novatel(raw_df):
    wynik = raw_df.copy()                                    # nie ruszamy argumentu

    # duplikaty (tylko pełne kopie wiersza = błąd eksportu)
    wynik = wynik.drop_duplicates().reset_index(drop=True)

    # teksty
    for kolumna in ["płeć", "województwo", "wielkość_miejscowości", "typ_umowy",
                    "zgoda_marketingowa", "kanał_pozyskania", "odejście_90d"]:
        wynik[kolumna] = wynik[kolumna].astype("string").str.strip().str.lower()
    wynik["województwo"] = wynik["województwo"].str.capitalize()
    wynik["kanał_pozyskania"] = wynik["kanał_pozyskania"].map(
        {"online": "online", "salon": "salon", "telefon": "telefon", "partner": "partner"})

    # daty i cel
    wynik["snapshot_date"] = pd.to_datetime(wynik["snapshot_date"], errors="coerce")
    wynik["data_zamknięcia_konta"] = pd.to_datetime(wynik["data_zamknięcia_konta"], errors="coerce")
    wynik["odejście_90d_bin"] = wynik["odejście_90d"].map({"nie": 0, "tak": 1})

    # wartości niepoprawne -> braki
    wynik.loc[wynik["wiek"].notna() & ~wynik["wiek"].between(18, 100), "wiek"] = np.nan
    wynik.loc[wynik["opłata_miesięczna_pln"] <= 0, "opłata_miesięczna_pln"] = np.nan
    wynik.loc[wynik["satysfakcja_1_5"].notna() & ~wynik["satysfakcja_1_5"].between(1, 5),
              "satysfakcja_1_5"] = np.nan
    wynik.loc[wynik["dochód_miesięczny_pln"] <= 0, "dochód_miesięczny_pln"] = np.nan

    # flagi braków (przed imputacją)
    wynik["brak_wieku"]       = wynik["wiek"].isna().astype(int)
    wynik["brak_satysfakcji"] = wynik["satysfakcja_1_5"].isna().astype(int)
    wynik["brak_dochodu"]     = wynik["dochód_miesięczny_pln"].isna().astype(int)
    wynik["brak_kanału"]      = wynik["kanał_pozyskania"].isna().astype(int)

    # imputacja
    wynik["wiek"] = wynik["wiek"].fillna(wynik["wiek"].median())
    wynik["satysfakcja_1_5"] = wynik["satysfakcja_1_5"].fillna(wynik["satysfakcja_1_5"].median())
    wynik["dochód_miesięczny_pln"] = wynik["dochód_miesięczny_pln"].fillna(
        wynik.groupby("wielkość_miejscowości")["dochód_miesięczny_pln"].transform("median"))
    wynik["opłata_miesięczna_pln"] = wynik["opłata_miesięczna_pln"].fillna(
        wynik.groupby("typ_umowy")["opłata_miesięczna_pln"].transform("median"))
    wynik["kanał_pozyskania"] = wynik["kanał_pozyskania"].fillna("nieznany")

    # cechy pochodne
    wynik["staż_lata"] = (wynik["staż_miesiące"] / 12).round(2)
    wynik["kontakt_reklamacja_razem"] = wynik["kontakty_z_obsługą_3m"] + wynik["reklamacje_6m"]
    wynik["aktywny_ostatnie_7d"] = (wynik["dni_od_ostatniego_logowania"] <= 7).astype(int)
    wynik["udział_opłaty_w_dochodzie_proc"] = (100 * wynik["opłata_miesięczna_pln"] / wynik["dochód_miesięczny_pln"]).round(2)
    wynik["opłata_po_rabacie_pln"] = (wynik["opłata_miesięczna_pln"] * (1 - wynik["rabat_proc"] / 100)).round(2)
    wynik["koszt_za_godzinę_pln"] = (wynik["opłata_miesięczna_pln"] /
                                     wynik["średnie_użycie_h_mies"].where(wynik["średnie_użycie_h_mies"] > 0)).round(2)

    return wynik


raw_kopia = raw.copy()
cleaned = clean_novatel(raw)

# funkcja daje ten sam wynik co kroki wykonane ręcznie
assert cleaned[df.columns].equals(df)
cleaned.head()

,customer_id,snapshot_date,wiek,płeć,województwo,wielkość_miejscowości,typ_umowy,staż_miesiące,opłata_miesięczna_pln,średnie_użycie_h_mies,...,brak_wieku,brak_satysfakcji,brak_dochodu,brak_kanału,staż_lata,kontakt_reklamacja_razem,aktywny_ostatnie_7d,udział_opłaty_w_dochodzie_proc,opłata_po_rabacie_pln,koszt_za_godzinę_pln
0,NT10234,2026-10-01,38.0,m,Mazowieckie,wieś,miesięczna,9,96.23,32.3,...,0,0,0,0,0.75,1,1,0.70,73.13,2.98
1,NT10110,2026-10-01,45.0,inna/nie podano,Mazowieckie,średnie miasto,12 miesięcy,67,100.62,63.9,...,0,0,0,0,5.58,4,0,0.53,87.54,1.57
2,NT10248,2026-10-01,36.0,m,Pomorskie,małe miasto,12 miesięcy,16,115.73,28.7,...,0,0,0,0,1.33,2,1,2.74,107.63,4.03
3,NT10009,2026-10-01,34.0,m,Dolnośląskie,średnie miasto,12 miesięcy,10,110.10,39.0,...,0,0,0,0,0.83,1,1,1.22,105.70,2.82
4,NT10093,2026-10-01,34.0,m,Mazowieckie,duże miasto,miesięczna,32,117.50,38.8,...,0,0,0,0,2.67,1,1,1.15,99.88,3.03


In [193]:
# argument nie został zmieniony
assert raw.equals(raw_kopia)

# unikalność
assert cleaned["customer_id"].is_unique

# zakresy
assert cleaned["wiek"].between(18, 100).all()
assert (cleaned["opłata_miesięczna_pln"] > 0).all()
assert cleaned["satysfakcja_1_5"].between(1, 5).all()
assert cleaned["rabat_proc"].between(0, 100).all()
assert (cleaned["staż_miesiące"] >= 0).all()
assert (cleaned["średnie_użycie_h_mies"] >= 0).all()
assert (cleaned["dochód_miesięczny_pln"] > 0).all()

# dozwolone kategorie
assert set(cleaned["odejście_90d"].unique()) <= {"tak", "nie"}
assert set(cleaned["kanał_pozyskania"].unique()) <= {"online", "salon", "telefon", "partner", "nieznany"}
assert set(cleaned["województwo"].unique()) == {"Mazowieckie", "Śląskie", "Dolnośląskie",
                                               "Wielkopolskie", "Pomorskie", "Małopolskie"}

# brak braków (poza datą zamknięcia konta, która jest pusta z założenia)
assert cleaned.drop(columns=["data_zamknięcia_konta"]).isna().sum().sum() == 0

# flagi braków zgodne z liczbą uzupełnionych wartości
assert cleaned["brak_wieku"].sum() == 13
assert cleaned["brak_satysfakcji"].sum() == 19
assert cleaned["brak_dochodu"].sum() == 28
assert cleaned["brak_kanału"].sum() == (cleaned["kanał_pozyskania"] == "nieznany").sum() == 7

# liczba rekordów (325 - 5 pełnych duplikatów)
assert len(cleaned) == len(raw) - raw.duplicated().sum() == 320

# wyciek informacji
assert "data_zamknięcia_konta" not in candidate_features
assert "odejście_90d" not in candidate_features and "odejście_90d_bin" not in candidate_features

# spójność: data zamknięcia konta jest tylko u klientów, którzy odeszli
assert (cleaned["data_zamknięcia_konta"].notna() == (cleaned["odejście_90d_bin"] == 1)).all()

print("Wszystkie testy zaliczone")

Wszystkie testy zaliczone


## 10. Eksport

Zapisz `cleaned` jako `novatel_lab2_clean.csv`, odczytaj zapisany plik ponownie i sprawdź, czy eksport zachował oczekiwaną strukturę oraz liczbę rekordów.

In [194]:
plik = Path("novatel_lab2_clean.csv")
cleaned.to_csv(plik, index=False, encoding="utf-8-sig", date_format="%Y-%m-%d")

sprawdzenie = pd.read_csv(plik, encoding="utf-8-sig")
print("Zapisane:", sprawdzenie.shape, "| przed zapisem:", cleaned.shape)

assert sprawdzenie.shape == cleaned.shape
assert list(sprawdzenie.columns) == list(cleaned.columns)
assert sprawdzenie["customer_id"].is_unique

# po odczycie daty są zwykłym tekstem, więc sprawdzamy typy
print(sprawdzenie[["snapshot_date", "data_zamknięcia_konta"]].dtypes)
print("Udział odejść:", sprawdzenie["odejście_90d_bin"].mean().round(3))

Zapisane: (320, 32) | przed zapisem: (320, 32)
snapshot_date            str
data_zamknięcia_konta    str
dtype: object
Udział odejść: 0.088


## Oddanie pracy

Zapisz notebook jako `Nazwisko_Imie_lab2.ipynb`. Oddaj notebook, oczyszczony CSV i uzupełnioną kartę pracy. Krótkie podsumowanie wpisz w ostatniej części karty.